# Python Environments — Stable Release Telemetry Dashboard

Runs queries filtered to **stable releases >= 1.24.0** only (even minor version, e.g. 1.24.x, 1.26.x, ...).
Pre-release builds (odd minor versions) are excluded.

**Prerequisites:**
1. `pip install -r requirements.txt`
2. `az login` (Azure CLI authentication)
3. `nbstripout --install` (one-time setup to auto-strip notebook outputs before commits)

In [ ]:
from initialize import initialize
from query_runner import run_kql, run_kql_file, load_kql_sections
from IPython.display import display, Markdown
import pandas as pd

# Show all rows and wrap long column text
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

client = initialize()
sections_09 = load_kql_sections("09-stable-manager-availability.kql")
print(f"Connected to Kusto. Loaded {len(sections_09)} sections from 09-stable-manager-availability.kql.")

---
## 1. MANAGER_REGISTRATION.FAILED — Total Failures by Manager × Version (stable >= 1.24.0)

Total failure count per manager per extension version, regardless of error type.
`MachinePct` = % of machines on that version that had any registration failure for this manager.
Same manager name appears in consecutive rows, sorted by version ascending.


In [ ]:
title, query = sections_09[0]
display(Markdown(f"### {title}"))
df_failures_by_version = run_kql(client, query)
display(df_failures_by_version)


---
## 2. SPAWN_TIMEOUT / PROCESS_CRASH / UNKNOWN Failures by Manager × Version (stable >= 1.24.0)

Merged query for the three most actionable error types. Use the `ErrorType` column to filter.
- `spawn_timeout` — PET hung; rising `MachinePct` in a new version → timeout regression
- `process_crash` — PET exited non-zero; rising `MachinePct` → crash regression
- `unknown` — unclassified errors; high counts in latest version → new paths need classification

Versions with zero failures for a given error type are absent (no row = no failures).


In [ ]:
title, query = sections_09[1]
display(Markdown(f"### {title}"))
df_error_types = run_kql(client, query)
display(df_error_types)


---
## 3. MANAGER.LAZY_INIT — Error Type Breakdown for pipenv / poetry / pyenv (stable >= 1.28.0)

From **1.28.0** (PR #1423), these three managers no longer emit `MANAGER_REGISTRATION.FAILED`.
Errors (`spawn_timeout`, `process_crash`, `unknown`, etc.) are now reported in `MANAGER.LAZY_INIT`
and only fire on **first actual use**, not at every startup.

**Denominator:** machines that sent *any* `MANAGER.LAZY_INIT` event for that manager+version
(success + tool_not_found + error). This means `MachinePct` = failure rate among machines that
actually engaged with the manager — closer to the pre-1.28.0 meaning than using all active
machines as denominator.

**Remaining caveat:** pre-1.28.0, registration ran on *every* VS Code launch for *all* machines.
Post-1.28.0, only users who opened the environment picker contribute. Rates are closer but still
not identical — a manager with very low engagement will show a noisier, potentially inflated rate.


In [ ]:
df_lazy_errors = run_kql_file(client, "10-stable-lazy-init-telemetry.kql")
display(df_lazy_errors)
